In [17]:
import pandas as pd
import pickle
import seaborn as sns
import matplotlib.pyplot as plt

import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error

In [18]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5050")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1739292931592, experiment_id='1', last_update_time=1739292931592, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [19]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [20]:
df_train = read_dataframe('../data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('../data/green_tripdata_2021-02.parquet')

df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [21]:
len(df_train), len(df_val)

(73908, 61921)

In [22]:
df_train.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration,PU_DO
0,2,2021-01-01 00:15:56,2021-01-01 00:19:52,N,1.0,43,151,1.0,1.01,5.5,...,0.00,0.0,None,0.3,6.80,2.0,1.0,0.00,3.933333,43_151
1,2,2021-01-01 00:25:59,2021-01-01 00:34:44,N,1.0,166,239,1.0,2.53,10.0,...,2.81,0.0,None,0.3,16.86,1.0,1.0,2.75,8.750000,166_239
2,2,2021-01-01 00:45:57,2021-01-01 00:51:55,N,1.0,41,42,1.0,1.12,6.0,...,1.00,0.0,None,0.3,8.30,1.0,1.0,0.00,5.966667,41_42
3,2,2020-12-31 23:57:51,2021-01-01 00:04:56,N,1.0,168,75,1.0,1.99,8.0,...,0.00,0.0,None,0.3,9.30,2.0,1.0,0.00,7.083333,168_75
7,2,2021-01-01 00:26:31,2021-01-01 00:28:50,N,1.0,75,75,6.0,0.45,3.5,...,0.96,0.0,None,0.3,5.76,1.0,1.0,0.00,2.316667,75_75


# Without sklearn's Pipeline

In [ ]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
print(f'{train_dicts[:5]=}')

# AUTOLOG
mlflow.sklearn.autolog()

X_train = dv.fit_transform(train_dicts)
print(f"{X_train[:5]=}")

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

with mlflow.start_run(run_name="lin-reg"):
    lr = LinearRegression()
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)

    # mean_squared_error(y_val, y_pred, squared=False)
    sklearn.metrics.root_mean_squared_error(y_val, y_pred)

In [ ]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

# Pipeline

In [23]:
from sklearn.pipeline import Pipeline

In [25]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

# AUTOLOG
mlflow.sklearn.autolog()

df_train = read_dataframe('../data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('../data/green_tripdata_2021-02.parquet')

df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

# dv = DictVectorizer()

pipeline = Pipeline([('dv', DictVectorizer()), ('lr', LinearRegression())])

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
val_dicts   = df_val[categorical + numerical].to_dict(orient='records')
target  = 'duration'
y_train = df_train[target].values
y_val   = df_val[target].values


# print(f'{train_dicts[:5]=}')
# X_train = dv.fit_transform(train_dicts)
# print(f"{X_train[:5]=}")

# X_val = dv.transform(val_dicts)


with mlflow.start_run(run_name="pipelined-lin-reg"):
    # lr = LinearRegression()
    pipeline.fit(train_dicts, y_train)

    y_pred = pipeline.predict(val_dicts)

    # mean_squared_error(y_val, y_pred, squared=False)
    sklearn.metrics.root_mean_squared_error(y_val, y_pred)

2025/02/12 17:38:45 WARNING mlflow.sklearn: Unrecognized dataset type <class 'list'>. Dataset logging skipped.
2025/02/12 17:38:46 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/02/12 17:38:46 WARNING mlflow.sklearn: Unrecognized dataset type <class 'list'>. Dataset logging skipped.


🏃 View run pipelined-lin-reg at: http://localhost:5050/#/experiments/1/runs/d33190cc82324c7f99f2761bff59a346
🧪 View experiment at: http://localhost:5050/#/experiments/1


In [26]:
mlflow.register_model('runs:/d33190cc82324c7f99f2761bff59a346/model', 'pipelined-lin-reg')

Successfully registered model 'pipelined-lin-reg'.
2025/02/12 17:46:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: pipelined-lin-reg, version 1
Created version '1' of model 'pipelined-lin-reg'.


<ModelVersion: aliases=[], creation_timestamp=1739378796827, current_stage='None', description='', last_updated_timestamp=1739378796827, name='pipelined-lin-reg', run_id='d33190cc82324c7f99f2761bff59a346', run_link='', source='mlflow-artifacts:/1/d33190cc82324c7f99f2761bff59a346/artifacts/model', status='READY', status_message=None, tags={}, user_id='', version='1'>